# Notebook 2 — Statistical Comparison of Anomaly Detection Models

This notebook performs the inferential statistical comparison between LGMMA-X and the six baseline models using the frozen chronological test set.

The analysis uses common chronological evaluation blocks as the paired statistical units. The same test sequences are assigned to the same evaluation block for every model, ensuring that model comparisons are paired on identical portions of the test set.

Positive performance differences indicate that LGMMA-X achieved a higher metric value than the corresponding baseline.

No model fitting, hyperparameter tuning, threshold selection, or test-set optimization is performed in this notebook.

## 1. Statistical Comparison Design

The frozen test set contains 21,858 sequences. To obtain repeated paired observations without creating artificial repetitions, the test sequences are divided chronologically into 10 approximately equal-sized evaluation blocks.

For each baseline and target metric:

\[
d_i = M_{\text{LGMMA-X},i} - M_{\text{Baseline},i}
\]

where \(i\) denotes the common chronological evaluation block.

The paired differences are tested for normality using the Shapiro-Wilk test.

- If the paired differences are approximately normal (`p >= 0.05`), a paired t-test is used.
- Otherwise, the Wilcoxon signed-rank test is used.

Because six baselines are compared for each metric, Holm-Bonferroni correction is applied separately within each metric at \(\alpha = 0.05\).

The raw reconstruction-error ranking ablation is threshold-free and therefore contributes only ROC-AUC and PR-AUC.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import shapiro, ttest_rel, wilcoxon
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

## 2. Paths and Frozen Evaluation Artifacts

All model outputs used here are frozen artifacts generated before statistical comparison.

The notebook does not retrain models or select thresholds.

In [ ]:
REPO_DIR = Path.cwd()

if REPO_DIR.name == "notebooks":
    SRC_DIR = REPO_DIR.parent
else:
    SRC_DIR = REPO_DIR / "src"

MODEL_DATA_DIR = SRC_DIR / "modeling" / "data"
COMPARISON_DIR = SRC_DIR / "baseline" / "statistical_comparison"

TEST_METADATA_PATH = (
    MODEL_DATA_DIR
    / "sequence_metadata"
    / "test_sequence_metadata.parquet"
)

TEST_LABELS_PATH = (
    MODEL_DATA_DIR
    / "evaluation"
    / "test_proxy_labels.csv"
)

GMM_SCORES_PATH = (
    MODEL_DATA_DIR
    / "gmm_results"
    / "test_anomaly_scores.csv"
)

GMM_THRESHOLD_PATH = (
    MODEL_DATA_DIR
    / "evaluation"
    / "selected_threshold.txt"
)

MODEL_FILES = {
    "LGMMA-X": GMM_SCORES_PATH,
    "ARIMA-GARCH": COMPARISON_DIR / "arima_garch_test_scores.csv",
    "Z-score": COMPARISON_DIR / "zscore_test_scores.csv",
    "Isolation Forest": COMPARISON_DIR / "isolation_forest_test_scores.csv",
    "One-Class SVM": COMPARISON_DIR / "one_class_svm_test_scores.csv",
    "LSTM-AE (Fixed Threshold)": COMPARISON_DIR / "lstm_fixed_test_scores.csv",
    "LSTM-AE (Raw Error Ranking)": COMPARISON_DIR / "lstm_rank_test_scores.csv",
}

BASELINES = list(MODEL_FILES)[1:]

BINARY_METRICS = [
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
]

SCORE_METRICS = [
    "roc_auc",
    "pr_auc",
]

TARGET_METRICS = BINARY_METRICS + SCORE_METRICS

N_BLOCKS = 10

STANDARD_COLUMNS = [
    "sequence_id",
    "segment_id",
    "start_timestamp",
    "end_timestamp",
    "anomaly_score",
    "proxy_anomaly",
]

BINARY_COLUMNS = STANDARD_COLUMNS + ["prediction"]

## 3. Load LGMMA-X Test Scores

LGMMA-X scores are reconstructed from the frozen GMM anomaly scores, test sequence metadata, proxy labels, and the frozen validation-calibrated threshold.

The threshold is not recalculated on the test set.

In [ ]:
def load_gmm_scores():
    metadata = pd.read_parquet(TEST_METADATA_PATH)[
        [
            "sequence_id",
            "segment_id",
            "start_timestamp",
            "end_timestamp",
        ]
    ]

    labels = pd.read_csv(TEST_LABELS_PATH)[
        ["sequence_id", "proxy_anomaly"]
    ]

    scores = pd.read_csv(GMM_SCORES_PATH)[
        ["sequence_id", "anomaly_score"]
    ]

    output = metadata.merge(
        scores,
        on="sequence_id",
        validate="one_to_one",
    )

    output = output.merge(
        labels,
        on="sequence_id",
        validate="one_to_one",
    )

    threshold = float(
        GMM_THRESHOLD_PATH.read_text(
            encoding="utf-8"
        ).strip()
    )

    output["prediction"] = (
        output["anomaly_score"] >= threshold
    ).astype(int)

    return output


lgmma_x = load_gmm_scores()

print(f"LGMMA-X test sequences: {len(lgmma_x):,}")
print(
    f"LGMMA-X threshold: "
    f"{float(GMM_THRESHOLD_PATH.read_text(encoding='utf-8').strip()):.6f}"
)

## 4. Load Standardized Baseline Score Files

Binary baselines must contain a frozen prediction generated using their previously selected validation threshold.

The LSTM-AE Raw Error Ranking ablation is intentionally treated differently because it is threshold-free and contains only continuous anomaly scores.

In [ ]:
def load_baseline_scores(model_name, path):
    output = pd.read_csv(path)

    required = set(STANDARD_COLUMNS)

    if model_name != "LSTM-AE (Raw Error Ranking)":
        required.add("prediction")

    missing = required - set(output.columns)

    if missing:
        raise ValueError(
            f"{model_name} is missing required columns: "
            f"{sorted(missing)}"
        )

    columns = (
        BINARY_COLUMNS
        if model_name != "LSTM-AE (Raw Error Ranking)"
        else STANDARD_COLUMNS
    )

    return output[columns].copy()


missing_files = [
    path
    for path in MODEL_FILES.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following frozen score files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )


score_tables = {
    "LGMMA-X": lgmma_x
}

for model, path in MODEL_FILES.items():
    if model == "LGMMA-X":
        continue

    score_tables[model] = load_baseline_scores(
        model,
        path,
    )

print(f"Loaded {len(score_tables)} models.")